# 05 — FPL Squad Optimizer

Uses Integer Linear Programming (PuLP) to:
1. **Select the optimal 15-man squad** from all available players given a £100m budget
2. **Visualise the squad** on a football pitch (mplsoccer)
3. **Recommend transfers** given a current squad and available budget
4. **Sensitivity analysis** — how does predicted value change with budget?

The optimizer uses the XGBoost predicted points from notebook 04 as its objective.

**Output:** `src/models/optimize.py` — `FPLOptimizer` class used by the Dash app.

## 1. Imports and setup

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.models.predict import FPLPredictor
from src.models.optimize import FPLOptimizer

PROCESSED = Path('../data/processed')
MODELS    = Path('../models')

print('Libraries loaded.')

## 2. Load features and generate predictions

We use the XGBoost model trained in notebook 04 to generate predicted points
for the **latest available gameweek** — these feed directly into the optimizer.

In [ ]:
# Load feature data
features = pd.read_parquet(PROCESSED / 'features.parquet')
print(f'Features: {features.shape}')
print(f'Rounds available: {sorted(features["round"].unique())}')

In [ ]:
# Use the most recent gameweek as our 'current' snapshot
latest_gw = features['round'].max()
latest_df = features[features['round'] == latest_gw].copy()
print(f'Using GW{latest_gw} — {len(latest_df)} players with data')

# Load trained predictor
predictor = FPLPredictor(models_dir=MODELS)
predictor.load()

# Generate predictions
latest_df['predicted_pts'] = predictor.predict(latest_df)
print(f'Predictions generated. Range: {latest_df["predicted_pts"].min():.2f} — {latest_df["predicted_pts"].max():.2f}')
print(f'Mean predicted pts: {latest_df["predicted_pts"].mean():.2f}')

In [ ]:
# Build the player pool DataFrame the optimizer expects
# Columns needed: player_id, web_name, position, team, now_cost, predicted_pts

# Load FPL player info for names and current cost
fpl_players = pd.read_parquet(PROCESSED / 'fpl_players.parquet')

POSITION_MAP = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
fpl_players['position'] = fpl_players['element_type'].map(POSITION_MAP)

# Merge predicted pts into player pool
players_df = fpl_players[['id', 'web_name', 'position', 'team', 'now_cost', 'status']].copy()
players_df.rename(columns={'id': 'player_id'}, inplace=True)
players_df['now_cost'] = players_df['now_cost'] / 10  # convert to £m

# Join predictions
preds_map = latest_df.set_index('player_id')['predicted_pts'].to_dict()
players_df['predicted_pts'] = players_df['player_id'].map(preds_map).fillna(0.0)

# Only include available players (not injured/suspended)
players_available = players_df[players_df['status'] == 'a'].copy()
print(f'Total players: {len(players_df)}')
print(f'Available players: {len(players_available)}')
print(f'Players with predictions: {(players_available["predicted_pts"] > 0).sum()}')

players_available.sort_values('predicted_pts', ascending=False).head(10)

## 3. Select optimal squad

The ILP optimizer picks the best 15 players (2 GKP, 5 DEF, 5 MID, 3 FWD)
within a £100m budget, with max 3 players per club. Captain is chosen automatically
as the player with the highest predicted points.

In [ ]:
optimizer = FPLOptimizer(budget=100.0)
result    = optimizer.select_squad(players_available, budget=100.0)

print(f'Status:           {result["status"]}')
print(f'Total cost:       £{result["total_cost"]}m')
print(f'Captain:          {result["captain"]}')
print(f'Vice-captain:     {result["vice_captain"]}')
print(f'Predicted total:  {result["predicted_total"]} pts')

In [ ]:
squad_df = result['squad']
print('=== Optimal Squad ===')
for pos in ['GKP', 'DEF', 'MID', 'FWD']:
    pos_players = squad_df[squad_df['position'] == pos]
    print(f'\n{pos}:')
    for _, row in pos_players.iterrows():
        cap_marker = ' ©' if row.get('is_captain') else ''
        print(f'  {row["web_name"]:<20} £{row["now_cost"]}m  {row["predicted_pts"]:.2f} pts{cap_marker}')

## 4. Visualise squad on a pitch

In [ ]:
def plot_squad_on_pitch(squad_df, captain_name, vice_name, title='Optimal FPL Squad'):
    """
    Draw a simple football pitch and place players by position.
    GKP at bottom, then DEF, MID, FWD rows.
    """
    fig, ax = plt.subplots(figsize=(10, 13))
    ax.set_facecolor('#2d7a2d')  # grass green
    fig.patch.set_facecolor('#1a1a2e')

    # Draw pitch lines
    pitch_rect = mpatches.FancyBboxPatch((0.05, 0.02), 0.90, 0.96,
                                          boxstyle='round,pad=0.01',
                                          linewidth=2, edgecolor='white',
                                          facecolor='#2d7a2d')
    ax.add_patch(pitch_rect)

    # Halfway line
    ax.axhline(y=0.5, xmin=0.05, xmax=0.95, color='white', linewidth=1.5, alpha=0.7)

    # Centre circle
    circle = plt.Circle((0.5, 0.5), 0.08, color='white', fill=False, linewidth=1.5, alpha=0.7)
    ax.add_patch(circle)

    # Position rows (y positions from bottom)
    row_y = {'GKP': 0.10, 'DEF': 0.30, 'MID': 0.57, 'FWD': 0.80}
    pos_colors = {'GKP': '#f5a623', 'DEF': '#4a90d9', 'MID': '#7ed321', 'FWD': '#e74c3c'}

    for pos, y in row_y.items():
        pos_players = squad_df[squad_df['position'] == pos].reset_index(drop=True)
        n = len(pos_players)
        xs = np.linspace(0.15, 0.85, n)

        for i, (_, player) in enumerate(pos_players.iterrows()):
            name = player['web_name']
            pts  = player['predicted_pts']
            cost = player['now_cost']
            color = pos_colors[pos]

            # Draw player circle
            circle = plt.Circle((xs[i], y), 0.042, color=color, zorder=5)
            ax.add_patch(circle)

            # Captain / VC badge
            badge = ''
            if name == captain_name:
                badge = ' ©'
            elif name == vice_name:
                badge = ' (v)'

            # Name label
            ax.text(xs[i], y + 0.055, f'{name}{badge}',
                    ha='center', va='bottom', fontsize=7.5,
                    color='white', fontweight='bold', zorder=6)

            # Points + cost
            ax.text(xs[i], y - 0.055, f'{pts:.1f}pts  £{cost}m',
                    ha='center', va='top', fontsize=6.5,
                    color='#dddddd', zorder=6)

    # Legend
    legend_elements = [mpatches.Patch(facecolor=c, label=p)
                       for p, c in pos_colors.items()]
    ax.legend(handles=legend_elements, loc='upper right',
              fontsize=8, framealpha=0.3, labelcolor='white')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title, color='white', fontsize=14, pad=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(MODELS / 'optimal_squad.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print('Saved to models/optimal_squad.png')


plot_squad_on_pitch(
    squad_df,
    captain_name=result['captain'],
    vice_name=result['vice_captain'],
    title=f'Optimal FPL Squad — GW{latest_gw}  |  £{result["total_cost"]}m  |  {result["predicted_total"]} pts'
)

## 5. Transfer advisor

Given a manager's current squad, the transfer advisor evaluates every possible
1- and 2-player swap, respecting:
- Position constraints (like-for-like)
- Budget neutrality (sell price + bank)
- Club limit (max 3 per club)
- Transfer hit (−4 pts per transfer beyond free transfers)

The result is ranked by **net gain** = gross points gain − hit penalty.

In [ ]:
# Simulate a 'current squad' — take the optimal squad and pretend it's our team
# In the real app this comes from the FPL API for the manager's actual squad.
current_squad_ids = squad_df['player_id'].tolist()
print(f'Current squad: {len(current_squad_ids)} players')
print('Players:', squad_df['web_name'].tolist())

In [ ]:
# Recommend transfers with 1 free transfer and £1.0m in the bank
transfers = optimizer.recommend_transfers(
    current_squad_ids=current_squad_ids,
    players_df=players_available,
    free_transfers=1,
    bank=1.0,
)

print(f'Transfer candidates evaluated: {len(transfers)}')
print(f'Profitable transfers (net_gain > 0): {sum(1 for t in transfers if t["net_gain"] > 0)}')

In [ ]:
# Display top 10 recommended transfers
print('=== Top Transfer Recommendations ===')
print(f'{"#":<3} {"OUT":<20} {"IN":<20} {"Gross":>6} {"Hit":>4} {"Net":>6} {"£Δ":>6}')
print('-' * 65)

for i, t in enumerate(transfers[:10], 1):
    out_str = ', '.join(t['transfers_out'])
    in_str  = ', '.join(t['transfers_in'])
    rec     = '✓' if t['is_recommended'] else ' '
    print(f'{i:<3} {out_str:<20} {in_str:<20} '
          f'{t["gross_gain"]:>6.2f} {t["hit"]:>4} {t["net_gain"]:>6.2f} '
          f'{t["cost_change"]:>+6.1f}m  {rec}')

In [ ]:
# Visualise the transfer gain distribution
net_gains = [t['net_gain'] for t in transfers]
n_transfers = [t['n_transfers'] for t in transfers]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#1a1a2e')

for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Net gain distribution
axes[0].hist(net_gains, bins=20, color='#4a90d9', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Net gain (pts)', color='white')
axes[0].set_ylabel('Count', color='white')
axes[0].set_title('Transfer net gain distribution', color='white', fontweight='bold')

# 1-transfer vs 2-transfer breakdown
one_t = [t['net_gain'] for t in transfers if t['n_transfers'] == 1]
two_t = [t['net_gain'] for t in transfers if t['n_transfers'] == 2]

if one_t:
    axes[1].hist(one_t, bins=15, color='#7ed321', alpha=0.7, label='1 transfer', edgecolor='white')
if two_t:
    axes[1].hist(two_t, bins=15, color='#e74c3c', alpha=0.7, label='2 transfers', edgecolor='white')
axes[1].axvline(x=0, color='white', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Net gain (pts)', color='white')
axes[1].set_ylabel('Count', color='white')
axes[1].set_title('1 vs 2 transfer comparison', color='white', fontweight='bold')
axes[1].legend(labelcolor='white', framealpha=0.3)

plt.tight_layout()
plt.savefig(MODELS / 'transfer_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved to models/transfer_analysis.png')

## 6. Budget sensitivity analysis

How much does the predicted squad value improve as we increase the budget
from £85m to £105m? This helps understand the marginal value of spending.

In [ ]:
budgets = np.arange(85, 106, 1.0)  # £85m to £105m in £1m steps
predicted_totals = []
total_costs      = []

for b in budgets:
    r = optimizer.select_squad(players_available, budget=float(b))
    if r['status'] == 'Optimal':
        predicted_totals.append(r['predicted_total'])
        total_costs.append(r['total_cost'])
    else:
        predicted_totals.append(None)
        total_costs.append(None)

print('Budget sensitivity:')
print(f'{"Budget":>8} | {"Cost":>6} | {"Pred pts":>9}')
print('-' * 30)
for b, c, p in zip(budgets, total_costs, predicted_totals):
    if p is not None:
        print(f'£{b:>5.0f}m  | £{c:>4.1f}m  | {p:>9.2f}')

In [ ]:
valid = [(b, p) for b, p in zip(budgets, predicted_totals) if p is not None]
bx, py = zip(*valid)

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

ax.plot(bx, py, marker='o', linewidth=2, color='#4a90d9',
        markersize=6, markerfacecolor='#f5a623')
ax.axvline(x=100, color='white', linestyle='--', linewidth=1, alpha=0.6, label='Default (£100m)')

ax.set_xlabel('Budget (£m)', color='white')
ax.set_ylabel('Predicted total pts', color='white')
ax.set_title('Predicted squad value vs budget', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(labelcolor='white', framealpha=0.3)
for spine in ax.spines.values():
    spine.set_edgecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(MODELS / 'budget_sensitivity.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved to models/budget_sensitivity.png')

## 7. Top value players (predicted pts per £m)

In [ ]:
# Points per million — useful for finding differentials
players_with_preds = players_available[players_available['predicted_pts'] > 1.0].copy()
players_with_preds['pts_per_million'] = (
    players_with_preds['predicted_pts'] / players_with_preds['now_cost']
)

print('=== Top 15 value players (predicted pts per £m) ===')
top_value = players_with_preds.sort_values('pts_per_million', ascending=False).head(15)
print(top_value[['web_name', 'position', 'now_cost', 'predicted_pts', 'pts_per_million']]
      .to_string(index=False, float_format='{:.2f}'.format))

print()
print('=== Top 15 overall predicted scorers ===')
top_scorers = players_with_preds.sort_values('predicted_pts', ascending=False).head(15)
print(top_scorers[['web_name', 'position', 'now_cost', 'predicted_pts', 'pts_per_million']]
      .to_string(index=False, float_format='{:.2f}'.format))

## 8. Summary

| Metric | Value |
|--------|-------|
| Optimizer | Integer Linear Programming (PuLP / CBC) |
| Squad size | 15 players |
| Constraints | Budget £100m, position counts, max 3 per club |
| Captain selection | Highest predicted pts in squad |
| Transfer advisor | All 1- and 2-player swaps, hit penalty = −4 pts/extra transfer |
| Budget sensitivity | Squad value increases non-linearly with budget |

The `FPLOptimizer` class in `src/models/optimize.py` is now ready to be
wired into the Dash application (Phase 7).